# Milestone 4: Persistent Memory & Deployment

Demonstrates the agent learning from HITL edits and using the Gmail API.

In [ ]:
from dotenv import load_dotenv
load_dotenv()

from memory import save_preference, load_all_preferences, get_interaction_history
print('Current preferences:')
print(load_all_preferences())

In [ ]:
# Simulate the HITL memory learning flow
# Step 1: Save a preference manually (simulates a prior HITL 'Edit' interaction)
save_preference('name_preference_bob', "Always refer to Bob Smith as 'Robert', not 'Bob'")
print('After saving preference:')
print(load_all_preferences())

In [ ]:
# Step 2: Now run the agent on an email referencing Bob — it should use 'Robert'
import uuid
from graph import graph
from state import AgentState

email = {
    'id': str(uuid.uuid4()),
    'sender': 'alice@company.com',
    'subject': 'Follow-up with Bob',
    'body': 'Hi, can you send a follow-up email to Bob Smith about the proposal?',
    'timestamp': '2024-01-22T10:00:00'
}

state: AgentState = {
    'email_input': email,
    'messages': [],
    'triage_result': None,
    'draft_response': None,
    'hitl_decision': None,
    'human_edit': None,
    'memory_context': None,
    'final_response': None,
}

config = {'configurable': {'thread_id': email['id']}}
# Note: in this notebook we skip the interactive HITL (it would block stdin)
# Set hitl_decision='approve' upfront to auto-approve
result = graph.invoke(state, config=config)
print('Draft response:')
print(result.get('draft_response', 'No draft generated'))

In [ ]:
# View interaction history
from memory import get_interaction_history
history = get_interaction_history(limit=5)
import json
print(json.dumps(history, indent=2))

In [ ]:
# Gmail integration test (requires credentials.json)
try:
    from gmail_service import fetch_unread_emails
    emails = fetch_unread_emails(max_results=3)
    for e in emails:
        print(f"From: {e['sender']} | Subject: {e['subject']}")
except Exception as ex:
    print(f'Gmail not configured yet: {ex}')
    print('Complete the OAuth setup in gmail_service.py to enable live email.')